# Phase 1 — Interpretability Harness Validation

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/PatrickJReed/cerberus-neuro/blob/main/notebooks/04_phase_1_harness.ipynb)

**Goal:** Run the full Phase 1 interpretability harness against the existing `cerberus-neuro-v0-baseline` checkpoint (val_acc 0.73). Validate every method end-to-end and produce inputs for the Phase 1 results report.

**Spec:** `docs/superpowers/specs/2026-05-12-argus-cells-design.md`, §4 + §5 + §6.
**Plan:** `docs/superpowers/plans/2026-05-12-argus-cells-phase-1-interpretability-harness.md`.
**Predecessor:** Phase 0 audit (PROCEED, 48 donors, crop budget ≫ target).

In [ ]:
# Install the argus_cells package code only (--no-deps) so pip never touches
# Colab's pre-built numpy / scipy / scikit-learn. Upgrading those mid-session
# breaks the numpy<->scipy ABI (the "_blas_supports_fpe" / "_center" ImportErrors).
!pip install -q --upgrade --no-deps git+https://github.com/PatrickJReed/argus-cells.git@main
# Only the two deps Colab may lack. Both are pure-Python and pull NO numpy-linked
# packages, so the scientific stack stays exactly as Colab shipped it. The data
# pipeline reads TIFFs with Pillow (already on Colab), so tifffile/imagecodecs
# are NOT needed; figures use matplotlib only, so seaborn is NOT needed.
!pip install -q boto3 huggingface_hub

In [ ]:
from huggingface_hub import login
from google.colab import drive, userdata

login(userdata.get("HF_TOKEN"))
drive.mount("/content/drive")

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

from argus_cells.data import build_manifest, subset_manifest, well_level_split, NeuroPaintingDataset
from argus_cells.model import BaselineDiseaseClassifier
from argus_cells.attribution import (
    compute_channel_ablation,
    compute_gradcam,
    compute_integrated_gradients,
)
from argus_cells.probes import parallel_probe_report
from argus_cells.analysis import (
    plot_channel_ablation_heatmap,
    plot_probe_comparison,
    stratify_channel_scores_by_cell_type,
)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"device: {DEVICE}")
if DEVICE == "cuda":
    print(f"gpu: {torch.cuda.get_device_name(0)}")

CACHE_DIR = Path("/content/drive/MyDrive/cerberus-neuro/cache")
CACHE_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
from huggingface_hub import hf_hub_download

# Phase 1's harness validates against the existing 0.73 baseline (best-epoch:
# 12 or 13 per docs/superpowers/results/2026-05-08-v0-phase-1-baseline-result.md).
HF_REPO = "patrickjreed/cerberus-neuro-v0-baseline"
CKPT_FILE = "epoch_013.pt"  # best-epoch val_acc 0.7311

ckpt_path = hf_hub_download(HF_REPO, CKPT_FILE)
ckpt = torch.load(ckpt_path, map_location="cpu", weights_only=False)

model = BaselineDiseaseClassifier(in_channels=6, n_classes=2, pretrained_encoder=False)
model.load_state_dict(ckpt["model"])
model = model.to(DEVICE).eval()
print(f"Loaded {HF_REPO}/{CKPT_FILE}")
print(f"Checkpoint epoch={ckpt.get('epoch')}, step={ckpt.get('step')}")
print(f"Parameter count: {model.parameter_count()}")

In [ ]:
# Cell 6 — Build a val-split dataloader. Reuse Phase 0.5/Phase 1 conventions:
# - 20x batches only (BATCHES_V0 default in data.py)
# - well-level split by (cell_type, line_condition), val_frac=0.2, seed=0
# - subset for harness validation: 200 wells_per_cell_type, 4 sites_per_well
from argus_cells.data import BATCHES_V0, CELL_TYPES, LINE_CONDITIONS

manifest = build_manifest(cache_dir=CACHE_DIR, batches=BATCHES_V0)
manifest = subset_manifest(manifest, wells_per_cell_type=200, sites_per_well=4, seed=0)
train_manifest, val_manifest = well_level_split(manifest, val_frac=0.2, seed=0)
print(f"train sites: {len(train_manifest):,}")
print(f"val sites:   {len(val_manifest):,}")

val_dataset = NeuroPaintingDataset(
    manifest=val_manifest,
    cache_dir=CACHE_DIR,
    crop_size=256,
    crops_per_site=10,
    min_cells_per_crop=3,
    shuffle=False,
    augment=False,
)
val_loader = torch.utils.data.DataLoader(
    val_dataset, batch_size=32, num_workers=4, persistent_workers=False
)

In [ ]:
# Cell 7 — Collect a CELL-TYPE-BALANCED batch of val crops for the attribution
# methods. val_manifest is grouped by cell type, so the old first-512 pass over a
# shuffle=False loader returned essentially one cell type (the others showed
# n_samples=0 in the stratification table). We instead bucket by (cell_type,
# line_condition) from a SHUFFLED pass and stop once every one of the 8 groups
# has TARGET_PER_GROUP crops, so the global ablation and the per-cell-type
# stratification are representative across all 4 cell types and both conditions.
TARGET_PER_GROUP = 64  # 4 cell_types x 2 conditions x 64 = up to 512 crops

collect_dataset = NeuroPaintingDataset(
    manifest=val_manifest,
    cache_dir=CACHE_DIR,
    crop_size=256,
    crops_per_site=10,
    min_cells_per_crop=3,
    shuffle=True,  # mix cell types so all 8 (cell_type, condition) groups fill together
    augment=False,
)
collect_loader = torch.utils.data.DataLoader(
    collect_dataset, batch_size=32, num_workers=4, persistent_workers=False
)

buckets = {
    (c, d): {"bf": [], "fluo": [], "ct": [], "cond": []}
    for c in range(len(CELL_TYPES))
    for d in range(len(LINE_CONDITIONS))
}


def groups_full():
    return all(len(b["bf"]) >= TARGET_PER_GROUP for b in buckets.values())


# Loop ends either when all 8 groups hit TARGET_PER_GROUP or the loader is
# exhausted (a sparse group simply contributes however many crops exist).
for bf, fluo, ct, cond in collect_loader:
    for j in range(bf.shape[0]):
        b = buckets[(int(ct[j]), int(cond[j]))]
        if len(b["bf"]) < TARGET_PER_GROUP:
            b["bf"].append(bf[j])
            b["fluo"].append(fluo[j])
            b["ct"].append(ct[j])
            b["cond"].append(cond[j])
    if groups_full():
        break

val_bf = torch.stack([t for b in buckets.values() for t in b["bf"]])
val_fluo = torch.stack([t for b in buckets.values() for t in b["fluo"]])
val_ct = torch.stack([t for b in buckets.values() for t in b["ct"]])
val_cond = torch.stack([t for b in buckets.values() for t in b["cond"]])

# Deterministic shuffle so the GradCAM subset (val_images[:32]) is also mixed,
# not all one (cell_type, condition) group.
perm = torch.randperm(val_ct.shape[0], generator=torch.Generator().manual_seed(0))
val_bf, val_fluo, val_ct, val_cond = val_bf[perm], val_fluo[perm], val_ct[perm], val_cond[perm]

val_images = torch.cat([val_bf, val_fluo], dim=1).to(DEVICE)
val_labels = val_cond.to(DEVICE)

print(f"collected val batch: {val_images.shape}")
for c in range(len(CELL_TYPES)):
    counts = {LINE_CONDITIONS[d]: len(buckets[(c, d)]["bf"]) for d in range(len(LINE_CONDITIONS))}
    print(f"  {CELL_TYPES[c]:>6}: {counts}")
print(f"cell_type distribution: {torch.bincount(val_ct, minlength=len(CELL_TYPES)).tolist()}")
print(f"condition distribution: {torch.bincount(val_cond, minlength=len(LINE_CONDITIONS)).tolist()}")

In [ ]:
# Cell 8 — Channel ablation. Per-channel accuracy drop across the whole val
# batch, AND per-sample per-channel confidence drop for stratification.
from argus_cells.attribution import compute_channel_ablation_per_sample

batch_result = compute_channel_ablation(model=model, images=val_images, labels=val_labels)
print("Per-channel accuracy drop (whole val batch):")
for i, ch in enumerate(["BF", "DNA", "Mito", "AGP", "ER", "RNA"]):
    print(f"  {ch:>4}: {batch_result.channel_scores[i].item():+.4f}")
print(f"baseline accuracy: {batch_result.metadata['baseline_accuracy']:.4f}")
print()

# Per-sample variant for stratification by cell type.
per_sample_result = compute_channel_ablation_per_sample(
    model=model, images=val_images, labels=val_labels,
)
print(f"per-sample channel_scores shape: {per_sample_result.channel_scores.shape}")

In [ ]:
# Cell 9 — GradCAM on the first 32 val crops. Full saliency maps are heavy;
# we keep a small representative subset for the figure.
subset_n = 32
gradcam_images = val_images[:subset_n]
gradcam_labels = val_labels[:subset_n]
gradcam_ct = val_ct[:subset_n]
gradcam_result = compute_gradcam(
    model=model,
    target_layer=model.encoder.layer4,
    images=gradcam_images,
    target_class=1,  # disease (deletion) class
)
print(f"GradCAM saliency shape: {gradcam_result.saliency.shape}")
print(f"channel_scores shape: {gradcam_result.channel_scores.shape}")

In [ ]:
# Cell 10 — IG on the same 32-crop subset. n_steps=32 is the sweet spot
# between accuracy and runtime on Colab L4 (~30s on GPU; ~5min on CPU).
ig_result = compute_integrated_gradients(
    model=model,
    images=gradcam_images,
    target_class=1,
    n_steps=32,
)
print(f"IG saliency shape: {ig_result.saliency.shape}")
print(f"channel_scores shape: {ig_result.channel_scores.shape}")
print(f"per-channel |sum| (mean over 32 samples):")
for i, ch in enumerate(["BF", "DNA", "Mito", "AGP", "ER", "RNA"]):
    print(f"  {ch:>4}: {ig_result.channel_scores[:, i].mean().item():+.3f}")

In [ ]:
# Cell 11 — Stratify per-sample channel ablation by cell type.
# Output: 4 (cell_type) x 6 (channel) = 24-row long-form table.
strat_df = stratify_channel_scores_by_cell_type(
    result=per_sample_result,
    cell_types=val_ct,
    cell_type_names=["stem", "progen", "neuron", "astro"],
    channel_names=["BF", "DNA", "Mito", "AGP", "ER", "RNA"],
)
print(strat_df.pivot(index="cell_type", columns="channel", values="mean_score").to_string())

In [ ]:
# Cell 12 — Extract frozen embeddings on train + val splits, each crop paired
# with its TRUE donor ID via the dataset's yield_donor flag, then fit two
# parallel probes (donor identity + disease). yield_donor=True makes the loader
# emit (bf, fluo, cell_type, line_condition, donor) so embedding i keeps donor i.
# This replaces the earlier positional-manifest approximation, which scrambled
# the donor labels and made the donor probe uninterpretable.
train_donor_dataset = NeuroPaintingDataset(
    manifest=train_manifest,
    cache_dir=CACHE_DIR,
    crop_size=256,
    crops_per_site=10,
    min_cells_per_crop=3,
    shuffle=True,
    augment=False,
    yield_donor=True,
)
val_donor_dataset = NeuroPaintingDataset(
    manifest=val_manifest,
    cache_dir=CACHE_DIR,
    crop_size=256,
    crops_per_site=10,
    min_cells_per_crop=3,
    shuffle=True,
    augment=False,
    yield_donor=True,
)
train_donor_loader = torch.utils.data.DataLoader(
    train_donor_dataset, batch_size=32, num_workers=4, persistent_workers=False
)
val_donor_loader = torch.utils.data.DataLoader(
    val_donor_dataset, batch_size=32, num_workers=4, persistent_workers=False
)


@torch.no_grad()
def collect_embeddings(loader, model, n_samples):
    """Iterate a yield_donor=True loader, returning embeddings paired with their
    true disease and donor labels. embedding i, disease i, and donor i all
    correspond to the same crop (no positional approximation)."""
    embs, diseases, donors = [], [], []
    collected = 0
    for bf, fluo, ct, cond, donor in loader:
        x = torch.cat([bf, fluo], dim=1).to(DEVICE)
        embs.append(model.extract_embedding(x).cpu().numpy())
        diseases.append(cond.cpu().numpy())
        donors.append(donor.cpu().numpy())
        collected += bf.shape[0]
        if collected >= n_samples:
            break
    return (
        np.concatenate(embs)[:n_samples],
        np.concatenate(diseases)[:n_samples],
        np.concatenate(donors)[:n_samples],
    )


# 1024 train + 256 val embeddings is plenty for a 48-way donor probe + 2-way disease probe.
train_emb, train_disease, train_donor_raw = collect_embeddings(train_donor_loader, model, n_samples=1024)
val_emb, val_disease, val_donor_raw = collect_embeddings(val_donor_loader, model, n_samples=256)

# Re-index observed donor IDs to contiguous [0, n_donors) for sklearn.
all_donors = sorted(set(train_donor_raw.tolist()) | set(val_donor_raw.tolist()))
donor_to_idx = {d: i for i, d in enumerate(all_donors)}
train_donor = np.array([donor_to_idx[d] for d in train_donor_raw], dtype="int64")
val_donor = np.array([donor_to_idx[d] for d in val_donor_raw], dtype="int64")
n_donors = len(all_donors)
print(f"train: {train_emb.shape}, val: {val_emb.shape}, n_donors observed: {n_donors}")

probe_report = parallel_probe_report(
    train_emb=train_emb, train_donor=train_donor, train_disease=train_disease.astype("int64"),
    val_emb=val_emb, val_donor=val_donor, val_disease=val_disease.astype("int64"),
    n_donors=n_donors,
)
print(f"donor probe val_acc: {probe_report['donor']['val_accuracy']:.4f} (baseline {probe_report['donor']['random_baseline']:.4f})")
print(f"disease probe val_acc: {probe_report['disease']['val_accuracy']:.4f} (baseline {probe_report['disease']['random_baseline']:.4f})")
print(f"ratio (donor/disease): {probe_report['ratio']:.3f}")

In [ ]:
# Cell 13 — Production figures for the Phase 1 results doc.
fig_heatmap = plot_channel_ablation_heatmap(
    df=strat_df,
    cell_type_order=["stem", "progen", "neuron", "astro"],
    channel_order=["BF", "DNA", "Mito", "AGP", "ER", "RNA"],
    title="Channel-ablation confidence drop per (cell type, channel) — Argus-RN34 v0",
)
plt.show()

fig_probe = plot_probe_comparison(
    report=probe_report,
    title=f"Donor probe (N={n_donors}) vs disease probe (N=2) — Argus-RN34 v0",
)
plt.show()

In [ ]:
# Cell 14 — Summary block for the Phase 1 results report. Paste into
# docs/superpowers/results/2026-05-12-phase-1-harness-result.md.
print("=" * 60)
print("PHASE 1 HARNESS RESULT SUMMARY")
print("=" * 60)
print()
print("Channel ablation (batch accuracy drop, sorted highest to lowest):")
order = batch_result.channel_scores.argsort(descending=True).tolist()
for i in order:
    ch = ["BF", "DNA", "Mito", "AGP", "ER", "RNA"][i]
    print(f"  {ch:>4}: drop = {batch_result.channel_scores[i].item():+.4f}")
print(f"  baseline_accuracy: {batch_result.metadata['baseline_accuracy']:.4f}")
print()
print("Cell-type stratified channel ablation (top channel per cell type):")
for ct in ["stem", "progen", "neuron", "astro"]:
    sub = strat_df[strat_df["cell_type"] == ct].sort_values("mean_score", ascending=False).head(2)
    pairs = ", ".join(f"{r.channel}={r.mean_score:+.3f}" for r in sub.itertuples())
    print(f"  {ct:>6}: {pairs}")
print()
print("Donor probe:")
print(f"  N donors observed: {n_donors}")
print(f"  donor val_acc: {probe_report['donor']['val_accuracy']:.4f}  (random baseline: {probe_report['donor']['random_baseline']:.4f})")
print(f"  disease val_acc: {probe_report['disease']['val_accuracy']:.4f}  (random baseline: 0.5)")
print(f"  ratio (donor/disease): {probe_report['ratio']:.3f}")
print(f"  interpretation: " + (
    "RED FLAG — donor info is at least as extractable as disease info" if probe_report['ratio'] >= 1.0
    else "OK — encoder retains less donor info than disease info"
))